# Experiment 2: TF-IDF + Ridge Classifier

This notebook implements a text classification pipeline using:
- **Vectorizer**: TF-IDF (Term Frequency-Inverse Document Frequency)
- **Model**: Ridge Classifier (L2 regularized linear classifier)

Ridge Classifier is similar to Logistic Regression but uses Ridge regression approach with class labels. It's often faster and can perform well on text data.

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import RidgeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, log_loss
from sklearn.pipeline import Pipeline
import os
import pickle
import json
from datetime import datetime

# Optuna (for hyperparameter tuning)
import optuna

In [2]:
DATA_PATH = "data/"
EXPERIMENT_PATH = "experiments/"
EXPERIMENT_NAME = "exp_2_tfidf_ridge"

In [3]:
## Setup Experiment Directory
experiment_dir = os.path.join(EXPERIMENT_PATH, EXPERIMENT_NAME)
os.makedirs(experiment_dir, exist_ok=True)
print(f"Experiment directory: {experiment_dir}")

Experiment directory: experiments/exp_2_tfidf_ridge


## Load Data

In [4]:
# Read the data
train = pd.read_csv(DATA_PATH + "train.csv")
test = pd.read_csv(DATA_PATH + "test.csv")

# Deduplicate rows
train = train.drop_duplicates().reset_index(drop=True)

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"\nTrain columns: {train.columns.tolist()}")
print(f"Test columns: {test.columns.tolist()}")

Train shape: (4212, 2)
Test shape: (1178, 2)

Train columns: ['text', 'label']
Test columns: ['id', 'text']


In [5]:
def create_pipeline(vectorizer_params=None, model_params=None):
    """
    Create a sklearn pipeline with vectorizer and model.
    This ensures proper fitting on train data only and prevents data leakage.
    
    Args:
        vectorizer_params: dict of TfidfVectorizer parameters
        model_params: dict of Ridge Classifier parameters
    
    Returns:
        sklearn.pipeline.Pipeline
    """
    if vectorizer_params is None:
        vectorizer_params = {
            'max_features': 10000,
            'ngram_range': (1, 2),
            'min_df': 2,
            'max_df': 0.95,
            'sublinear_tf': True
        }
    
    if model_params is None:
        model_params = {
            'alpha': 1.0,
            'random_state': 42
        }
    
    pipeline = Pipeline([
        ('vectorizer', TfidfVectorizer(**vectorizer_params)),
        ('model', RidgeClassifier(**model_params))
    ])
    
    return pipeline

# Initialize pipeline parameters
vectorizer_params = {
    'max_features': 10000,
    'ngram_range': (1, 2),
    'min_df': 2,
    'max_df': 0.95,
    'sublinear_tf': True
}

model_params = {
    'alpha': 1.0,
    'random_state': 42
}

print("Pipeline configuration:")
print(f"Vectorizer: TF-IDF with {vectorizer_params}")
print(f"Model: Ridge Classifier with {model_params}")

Pipeline configuration:
Vectorizer: TF-IDF with {'max_features': 10000, 'ngram_range': (1, 2), 'min_df': 2, 'max_df': 0.95, 'sublinear_tf': True}
Model: Ridge Classifier with {'alpha': 1.0, 'random_state': 42}


## Create Pipeline Function

Using sklearn Pipeline ensures:
1. No data leakage (vectorizer fits only on training data per fold)
2. Clean, reproducible code
3. Easy model deployment

In [6]:
# Prepare data
X_train_text = train['text'].values
y_train = train['label'].values
X_test_text = test['text'].values

print(f"Train samples: {len(X_train_text)}")
print(f"Test samples: {len(X_test_text)}")

Train samples: 4212
Test samples: 1178


## Prepare Data for Training

In [7]:
# 5-Fold Cross Validation
n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

# Storage for models and metrics
fold_models = []
fold_metrics = []
fold_predictions = np.zeros(len(test))
all_feature_names = []  # Store feature names from each fold

## Cross-Validation Setup

In [8]:
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_text, y_train), 1):
    print(f"\nFold {fold}/{n_folds}")
    print("-"*60)
    
    # Split data - IMPORTANT: Split before any fitting
    X_fold_train, X_fold_val = X_train_text[train_idx], X_train_text[val_idx]
    y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]
    
    # Create fresh pipeline for this fold
    pipeline = create_pipeline(vectorizer_params, model_params)
    
    # Fit pipeline on training fold only (TF-IDF learns from train only)
    pipeline.fit(X_fold_train, y_fold_train)
    
    # Validation predictions
    y_val_pred = pipeline.predict(X_fold_val)
    # Ridge Classifier has decision_function instead of predict_proba
    y_val_scores = pipeline.decision_function(X_fold_val)
    
    # Calculate metrics
    # Note: For ROC AUC and log loss, we use decision_function scores
    metrics = {
        'fold': fold,
        'accuracy': accuracy_score(y_fold_val, y_val_pred),
        'precision': precision_score(y_fold_val, y_val_pred),
        'recall': recall_score(y_fold_val, y_val_pred),
        'f1': f1_score(y_fold_val, y_val_pred),
        'roc_auc': roc_auc_score(y_fold_val, y_val_scores),
        'train_size': len(train_idx),
        'val_size': len(val_idx)
    }
    
    fold_metrics.append(metrics)
    
    # Print metrics
    print(f"Accuracy:  {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall:    {metrics['recall']:.4f}")
    print(f"F1 Score:  {metrics['f1']:.4f}")
    print(f"ROC AUC:   {metrics['roc_auc']:.4f}")
    
    # Test predictions using this fold's pipeline
    test_scores = pipeline.decision_function(X_test_text)
    fold_predictions += test_scores / n_folds
    
    # Save pipeline (includes both vectorizer and model)
    pipeline_path = os.path.join(experiment_dir, f'pipeline_fold_{fold}.pkl')
    with open(pipeline_path, 'wb') as f:
        pickle.dump(pipeline, f)
    
    # Save feature names from this fold's vectorizer
    feature_names = pipeline.named_steps['vectorizer'].get_feature_names_out().tolist()
    all_feature_names.append({
        'fold': fold,
        'n_features': len(feature_names),
        'features': feature_names
    })
    
    fold_models.append({
        'fold': fold,
        'pipeline_path': pipeline_path,
        'metrics': metrics,
        'n_features': len(feature_names)
    })

print("="*60)
print("Cross Validation Complete!")
print("="*60)


Fold 1/5
------------------------------------------------------------
Accuracy:  0.9739
Precision: 0.9724
Recall:    0.9527
F1 Score:  0.9625
ROC AUC:   0.9963

Fold 2/5
------------------------------------------------------------
Accuracy:  0.9656
Precision: 0.9785
Recall:    0.9223
F1 Score:  0.9496
ROC AUC:   0.9952

Fold 3/5
------------------------------------------------------------
Accuracy:  0.9703
Precision: 0.9688
Recall:    0.9458
F1 Score:  0.9571
ROC AUC:   0.9961

Fold 4/5
------------------------------------------------------------
Accuracy:  0.9667
Precision: 0.9786
Recall:    0.9257
F1 Score:  0.9514
ROC AUC:   0.9973

Fold 5/5
------------------------------------------------------------
Accuracy:  0.9739
Precision: 0.9824
Recall:    0.9426
F1 Score:  0.9621
ROC AUC:   0.9978
Cross Validation Complete!


## Train Models with 5-Fold Cross-Validation

In [9]:
# Calculate average metrics and identify best fold
metrics_df = pd.DataFrame(fold_metrics)
print("\nPer-Fold Metrics:")
print(metrics_df.to_string(index=False))

print("\n" + "="*60)
print("Average Metrics Across Folds:")
print("-"*60)
for col in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    mean_val = metrics_df[col].mean()
    std_val = metrics_df[col].std()
    print(f"{col.upper():12s}: {mean_val:.4f} ± {std_val:.4f}")

# Identify best fold (highest ROC AUC)
best_fold_idx = metrics_df['roc_auc'].idxmax()
best_fold = metrics_df.loc[best_fold_idx, 'fold']
best_roc_auc = metrics_df.loc[best_fold_idx, 'roc_auc']

print(f"BEST FOLD: {int(best_fold)} (ROC AUC: {best_roc_auc:.4f})")


Per-Fold Metrics:
 fold  accuracy  precision   recall       f1  roc_auc  train_size  val_size
    1  0.973903   0.972414 0.952703 0.962457 0.996325        3369       843
    2  0.965599   0.978495 0.922297 0.949565 0.995207        3369       843
    3  0.970309   0.968750 0.945763 0.957118 0.996071        3370       842
    4  0.966746   0.978571 0.925676 0.951389 0.997327        3370       842
    5  0.973872   0.982394 0.942568 0.962069 0.997803        3370       842

Average Metrics Across Folds:
------------------------------------------------------------
ACCURACY    : 0.9701 ± 0.0039
PRECISION   : 0.9761 ± 0.0055
RECALL      : 0.9378 ± 0.0132
F1          : 0.9565 ± 0.0059
ROC_AUC     : 0.9965 ± 0.0010
BEST FOLD: 5 (ROC AUC: 0.9978)


## Evaluate Cross-Validation Results

In [10]:
# Save metrics summary
metrics_summary = {
    'experiment_name': EXPERIMENT_NAME,
    'model_type': 'Ridge Classifier',
    'vectorizer': 'TF-IDF',
    'vectorizer_params': vectorizer_params,
    'model_params': model_params,
    'n_folds': n_folds,
    'best_fold': int(best_fold),
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'average_metrics': {
        'accuracy': float(metrics_df['accuracy'].mean()),
        'accuracy_std': float(metrics_df['accuracy'].std()),
        'precision': float(metrics_df['precision'].mean()),
        'precision_std': float(metrics_df['precision'].std()),
        'recall': float(metrics_df['recall'].mean()),
        'recall_std': float(metrics_df['recall'].std()),
        'f1': float(metrics_df['f1'].mean()),
        'f1_std': float(metrics_df['f1'].std()),
        'roc_auc': float(metrics_df['roc_auc'].mean()),
        'roc_auc_std': float(metrics_df['roc_auc'].std())
    },
    'fold_metrics': fold_metrics,
    'fold_models': fold_models
}

# Save to JSON
with open(os.path.join(experiment_dir, 'metrics.json'), 'w') as f:
    json.dump(metrics_summary, f, indent=2)

print(f"\nMetrics saved to {os.path.join(experiment_dir, 'metrics.json')}")


Metrics saved to experiments/exp_2_tfidf_ridge/metrics.json


## Save Metrics Summary

In [11]:
# Create submission file
# Normalize decision function scores to [0, 1] range using sigmoid
from scipy.special import expit

# Apply sigmoid to convert scores to probabilities
fold_probabilities = expit(fold_predictions)

submission = pd.DataFrame({
    'id': test['id'],
    'probability': fold_probabilities
})

submission_path = os.path.join(experiment_dir, f'{EXPERIMENT_NAME}_submission.csv')
submission.to_csv(submission_path, index=False)

print(f"\nSubmission file created: {submission_path}")
print(f"Shape: {submission.shape}")
print(f"\nFirst few predictions:")
print(submission.head(10))


Submission file created: experiments/exp_2_tfidf_ridge/exp_2_tfidf_ridge_submission.csv
Shape: (1178, 2)

First few predictions:
   id  probability
0   0     0.342368
1   1     0.301130
2   2     0.287774
3   3     0.339512
4   4     0.301567
5   5     0.272805
6   6     0.266602
7   7     0.665057
8   8     0.572972
9   9     0.350932


## Create Submission File

Note: Ridge Classifier outputs decision function scores, not probabilities. We'll normalize them to [0, 1] range for submission.

In [12]:
# Save all feature names from each fold
feature_names_path = os.path.join(experiment_dir, 'feature_names_all_folds.json')
with open(feature_names_path, 'w') as f:
    json.dump(all_feature_names, f, indent=2)

print(f"\nFeature names from all folds saved to: {feature_names_path}")
print(f"Feature counts per fold:")
for fold_features in all_feature_names:
    print(f"  Fold {fold_features['fold']}: {fold_features['n_features']} features")


Feature names from all folds saved to: experiments/exp_2_tfidf_ridge/feature_names_all_folds.json
Feature counts per fold:
  Fold 1: 10000 features
  Fold 2: 10000 features
  Fold 3: 10000 features
  Fold 4: 10000 features
  Fold 5: 10000 features


## Save Feature Names

In [13]:
# Summary of saved artifacts
print("EXPERIMENT SUMMARY")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Directory: {experiment_dir}")
print(f"\nSaved Artifacts:")
print(f"  - 5 pipeline files (pipeline_fold_1.pkl to pipeline_fold_5.pkl)")
print(f"    Each pipeline contains: TF-IDF vectorizer + Ridge Classifier")
print(f"  - metrics.json (detailed metrics for all folds)")
print(f"  - feature_names_all_folds.json (features from each fold)")
print(f"  - {EXPERIMENT_NAME}_submission.csv (averaged predictions)")
print(f"\nBest Model: Fold {int(best_fold)} (ROC AUC: {best_roc_auc:.4f})")
print(f"\nTo load and use a pipeline:")
print(f"  with open('pipeline_fold_{int(best_fold)}.pkl', 'rb') as f:")
print(f"      pipeline = pickle.load(f)")
print(f"  scores = pipeline.decision_function(texts)")
print(f"  probabilities = expit(scores)  # Convert to probabilities")

EXPERIMENT SUMMARY
Experiment: exp_2_tfidf_ridge
Directory: experiments/exp_2_tfidf_ridge

Saved Artifacts:
  - 5 pipeline files (pipeline_fold_1.pkl to pipeline_fold_5.pkl)
    Each pipeline contains: TF-IDF vectorizer + Ridge Classifier
  - metrics.json (detailed metrics for all folds)
  - feature_names_all_folds.json (features from each fold)
  - exp_2_tfidf_ridge_submission.csv (averaged predictions)

Best Model: Fold 5 (ROC AUC: 0.9978)

To load and use a pipeline:
  with open('pipeline_fold_5.pkl', 'rb') as f:
      pipeline = pickle.load(f)
  scores = pipeline.decision_function(texts)
  probabilities = expit(scores)  # Convert to probabilities


## Experiment Summary

In [14]:
def optuna_objective(trial, X, y, cv_splits=3, random_state=42):
    # TF-IDF hyperparameters
    max_features = trial.suggest_categorical('max_features', [5000, 10000, 20000, None])
    ngram_min = trial.suggest_int('ngram_min', 1, 3)
    ngram_max = trial.suggest_int('ngram_max', ngram_min, 5)
    min_df = trial.suggest_int('min_df', 1, 5)
    max_df = trial.suggest_float('max_df', 0.7, 1.0)
    sublinear_tf = trial.suggest_categorical('sublinear_tf', [True, False])

    # Ridge Classifier hyperparameters
    alpha = trial.suggest_float('alpha', 1e-4, 100, log=True)
    # Only use solvers that support sparse matrices (TF-IDF output)
    # Exclude 'svd', 'saga' and 'cholesky' as they don't support sparse data
    solver = trial.suggest_categorical('solver', ['auto', 'lsqr', 'sparse_cg', 'sag'])

    vectorizer_params = {
        'max_features': max_features,
        'ngram_range': (ngram_min, ngram_max),
        'min_df': min_df,
        'max_df': max_df,
        'sublinear_tf': sublinear_tf
    }
    model_params = {'alpha': alpha, 'solver': solver, 'random_state': random_state}

    skf_local = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)
    roc_aucs = []

    for train_idx, val_idx in skf_local.split(X, y):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        # Create and fit pipeline
        pipeline = Pipeline([
            ('vectorizer', TfidfVectorizer(**{k: v for k, v in vectorizer_params.items() if v is not None})),
            ('model', RidgeClassifier(**model_params))
        ])

        pipeline.fit(X_tr, y_tr)
        y_val_scores = pipeline.decision_function(X_val)
        roc_aucs.append(roc_auc_score(y_val, y_val_scores))

        # pruning (optional)
        trial.report(np.mean(roc_aucs), len(roc_aucs))
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(roc_aucs))

---

## Optional: Hyperparameter Tuning with Optuna

The following cells use Optuna to find optimal hyperparameters. This is optional and can take significant time.

In [15]:
# Run Optuna study (small number of trials by default; increase for production)
study_path = os.path.join(experiment_dir, 'optuna_study.pkl')

study = optuna.create_study(direction='maximize', study_name=EXPERIMENT_NAME)
try:
    study.optimize(lambda trial: optuna_objective(trial, X_train_text, y_train, cv_splits=3), n_trials=50)
except KeyboardInterrupt:
    print('Study interrupted by user')

print(f"Best trial value: {study.best_value}")
print(f"Best trial params: {study.best_params}")

# Save study
with open(study_path, 'wb') as f:
    pickle.dump(study, f)

print(f"Optuna study saved to {study_path}")

[I 2025-11-07 19:52:09,991] A new study created in memory with name: exp_2_tfidf_ridge
[I 2025-11-07 19:52:11,179] Trial 0 finished with value: 0.973427472355383 and parameters: {'max_features': 5000, 'ngram_min': 2, 'ngram_max': 2, 'min_df': 3, 'max_df': 0.8990442630101358, 'sublinear_tf': False, 'alpha': 14.7785795244036, 'solver': 'sag'}. Best is trial 0 with value: 0.973427472355383.
[I 2025-11-07 19:52:18,009] Trial 1 finished with value: 0.9502096307692991 and parameters: {'max_features': 10000, 'ngram_min': 3, 'ngram_max': 5, 'min_df': 1, 'max_df': 0.9647654281471442, 'sublinear_tf': True, 'alpha': 0.9750562045517094, 'solver': 'sag'}. Best is trial 0 with value: 0.973427472355383.
[I 2025-11-07 19:52:23,070] Trial 2 finished with value: 0.981016336282043 and parameters: {'max_features': 20000, 'ngram_min': 2, 'ngram_max': 4, 'min_df': 3, 'max_df': 0.7218774655911288, 'sublinear_tf': False, 'alpha': 6.506915073929734, 'solver': 'auto'}. Best is trial 2 with value: 0.981016336282

Best trial value: 0.9961287516634271
Best trial params: {'max_features': 20000, 'ngram_min': 1, 'ngram_max': 2, 'min_df': 4, 'max_df': 0.804325072647005, 'sublinear_tf': True, 'alpha': 0.2905058487907762, 'solver': 'sparse_cg'}
Optuna study saved to experiments/exp_2_tfidf_ridge/optuna_study.pkl


In [16]:
# Retrain best pipeline on full training data using best params
best_params = study.best_params

# Extract tfidf params from best_params
vectorizer_best = {
    'max_features': best_params.get('max_features', None),
    'ngram_range': (best_params.get('ngram_min', 1), best_params.get('ngram_max', 2)),
    'min_df': best_params.get('min_df', 2),
    'max_df': best_params.get('max_df', 0.95),
    'sublinear_tf': best_params.get('sublinear_tf', True)
}

model_best = {
    'alpha': best_params.get('alpha', 1.0),
    'solver': best_params.get('solver', 'auto'),
    'random_state': 42
}

final_pipeline = Pipeline([
    ('vectorizer', TfidfVectorizer(**{k: v for k, v in vectorizer_best.items() if v is not None})),
    ('model', RidgeClassifier(**model_best))
])

final_pipeline.fit(X_train_text, y_train)

# Save final pipeline and vectorizer separately
final_pipeline_path = os.path.join(experiment_dir, 'pipeline_best_full.pkl')
with open(final_pipeline_path, 'wb') as f:
    pickle.dump(final_pipeline, f)

vectorizer_path = os.path.join(experiment_dir, 'tfidf_vectorizer_best.pkl')
with open(vectorizer_path, 'wb') as f:
    pickle.dump(final_pipeline.named_steps['vectorizer'], f)

print(f"Saved final pipeline to {final_pipeline_path}")
print(f"Saved best TF-IDF vectorizer to {vectorizer_path}")

# Create final submission using final pipeline
from scipy.special import expit
final_test_scores = final_pipeline.decision_function(X_test_text)
final_test_proba = expit(final_test_scores)  # Convert to probabilities

submission_final = pd.DataFrame({'id': test['id'], 'probability': final_test_proba})
submission_final_path = os.path.join(experiment_dir, f'{EXPERIMENT_NAME}_submission_optuna.csv')
submission_final.to_csv(submission_final_path, index=False)
print(f"Final submission saved to {submission_final_path}")

Saved final pipeline to experiments/exp_2_tfidf_ridge/pipeline_best_full.pkl
Saved best TF-IDF vectorizer to experiments/exp_2_tfidf_ridge/tfidf_vectorizer_best.pkl
Final submission saved to experiments/exp_2_tfidf_ridge/exp_2_tfidf_ridge_submission_optuna.csv


In [17]:
# Save Optuna metadata to metrics.json
metrics_path = os.path.join(experiment_dir, 'metrics.json')
with open(metrics_path, 'r') as f:
    metrics_data = json.load(f)

metrics_data['optuna_study'] = {
    'best_value': float(study.best_value),
    'best_params': study.best_params,
    'n_trials': len(study.trials),
    'study_path': study_path
}

with open(metrics_path, 'w') as f:
    json.dump(metrics_data, f, indent=2)

print(f"Updated {metrics_path} with Optuna results")

Updated experiments/exp_2_tfidf_ridge/metrics.json with Optuna results
